# Solució — Sessió 6

### Mini-repte: llegir, entendre i escriure MIDI amb `mido`

**Per què a Colab i no a Thonny?** Aquí treballem amb *fitxers* `.mid`: no hi ha cap port ni cap rellotge en temps real. Colab ens deixa dibuixar el piano roll i escoltar el resultat al moment. Quan toquem ports MIDI reals (teclats, DAW), tornarem a Thonny.

**Què faràs:**
1. Comptar notes d'un fitxer
2. Trobar-ne el registre (nota més greu i més aguda)
3. Convertir els *delta times* a temps absolut — el concepte clau d'avui
4. Escriure una seqüència pròpia **amb un silenci i un acord**
5. Transposar-la i muntar un duo

Completa els `# TODO`. Després de cada funció hi ha una cel·la **🔊 prova-ho** (per entendre què fa) i una d'**AUTOTEST** (per comprovar-ho). No esborris les cel·les d'autotest.

In [ ]:
!apt-get install -y fluidsynth fluid-soundfont-gm -q > /dev/null
!pip install mido pretty_midi pyfluidsynth -q
import mido
import pretty_midi
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/04_midi/sessio06_estructura_midi/recursos/midi/"
urllib.request.urlretrieve(base_url + "example_scale.mid", "example_scale.mid")
print("Fitxers carregats")

### Eines (ja fetes — no cal tocar-les)
- `escolta(fitxer)`: converteix un `.mid` a àudio i el reprodueix.
- `tempo_de(mid)`: busca el missatge `set_tempo` del track de metadades.
- `dibuixa_piano_roll(veus, etiquetes)`: dibuixa una o més llistes de notes `(nota, velocity, inici_s, fi_s)`. **Aquestes llistes les generarà la teva funció de la Part 3.**

In [ ]:
NOMS = ['Do','Do#','Re','Re#','Mi','Fa','Fa#','Sol','Sol#','La','La#','Si']

def nom_nota(n):
    return f"{NOMS[n % 12]}{n // 12 - 1}"

def escolta(fitxer):
    pm = pretty_midi.PrettyMIDI(fitxer)
    total = 0
    for inst in pm.instruments:
        total = total + len(inst.notes)
    if total == 0:
        print(f"⚠️ {fitxer} no conté cap nota audible (notes de durada 0?)")
        return
    display(Audio(pm.fluidsynth(fs=44100), rate=44100))

def tempo_de(mid):
    tempo = 500000                       # valor per defecte de MIDI (120 bpm)
    for msg in mid.tracks[0]:
        if msg.type == 'set_tempo':
            tempo = msg.tempo
    return tempo

def dibuixa_piano_roll(veus, etiquetes, titol="Piano roll"):
    colors = ['steelblue', 'coral', 'seagreen', 'orchid']
    fig, ax = plt.subplots(figsize=(12, 4))
    mes_greu = 127
    mes_aguda = 0
    for k in range(len(veus)):
        etiqueta = etiquetes[k]                 # només la primera nota de cada veu porta etiqueta
        for nota, vel, inici, fi in veus[k]:
            ax.barh(nota, fi - inici, left=inici, height=0.7,
                    color=colors[k % 4], alpha=0.35 + 0.65 * vel / 127, label=etiqueta)
            etiqueta = None
            mes_greu = min(mes_greu, nota)
            mes_aguda = max(mes_aguda, nota)
    if mes_aguda == 0:
        print("No hi ha notes per dibuixar")
        return
    files = list(range(mes_greu - 1, mes_aguda + 2))
    noms = []
    for n in files:
        noms.append(nom_nota(n))
    ax.set_yticks(files)
    ax.set_yticklabels(noms, fontsize=7)
    ax.set_xlabel("Temps (s)")
    ax.set_title(titol)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

## Part 1 — Comptar notes

Implementa `compta_notes`: rep un track de `mido` i retorna quantes notes conté (missatges `note_on` amb velocity > 0).

*Per què `velocity > 0`?* Molts fitxers tanquen les notes amb un `note_on` de velocity 0 en lloc d'un `note_off`. Si no ho filtres, comptaràs cada nota dues vegades.

In [ ]:
def compta_notes(track):
    n = 0
    for msg in track:
        if msg.type == 'note_on' and msg.velocity > 0:
            n += 1
    return n

# ── Variants que poden proposar (totes retornen 16 amb la melodia) ──
#
# A) Índex + comptador, gairebé C:
#     n = 0
#     for i in range(len(track)):
#         msg = track[i]
#         if msg.type == 'note_on':
#             if msg.velocity > 0:
#                 n = n + 1
#     return n
#
# B) Filtrar a una llista i comptar-la:
#     notes = []
#     for msg in track:
#         if msg.type == 'note_on' and msg.velocity > 0:
#             notes.append(msg)
#     return len(notes)
#     # compacta: return len([msg for msg in track if msg.type == 'note_on' and msg.velocity > 0])
#
# C) Idiomàtica (generador + sum):
#     return sum(1 for msg in track if msg.type == 'note_on' and msg.velocity > 0)
#
# ⚠️ PARANY (verificat): invertir l'ordre de la condició
#     if msg.velocity > 0 and msg.type == 'note_on':
# peta amb AttributeError: 'MetaMessage' object has no attribute 'velocity'.
# El primer missatge del track és un meta (track_name), i els meta, control_change
# i program_change no tenen velocity. Amb el type primer, el `and` ja no avalua
# la segona part. Els dos `if` niats de la variant A protegeixen igual.

### 🔊 Prova-ho
Escolta el fitxer i compta les notes d'oïda. Quadra amb el que diu la teva funció?

In [ ]:
mid = mido.MidiFile('example_scale.mid')
escolta('example_scale.mid')
for i, track in enumerate(mid.tracks):
    print(f"Track {i} '{track.name}': {compta_notes(track)} notes")

In [ ]:
# AUTOTEST 1
mid = mido.MidiFile('example_scale.mid')
assert compta_notes(mid.tracks[1]) is not None, "La funció retorna None"
assert compta_notes(mid.tracks[0]) == 0, "El track de metadades no té notes"
assert compta_notes(mid.tracks[1]) == 16, f"La melodia té 16 notes, la funció en diu {compta_notes(mid.tracks[1])}"
assert compta_notes(mid.tracks[2]) == 8, f"El baix té 8 notes, la funció en diu {compta_notes(mid.tracks[2])}"

t = mido.MidiTrack()
t.append(mido.Message('note_on', note=60, velocity=80, time=0))
t.append(mido.Message('note_on', note=60, velocity=0, time=480))   # aquest TANCA la nota
assert compta_notes(t) == 1, "Un note_on amb velocity 0 tanca una nota: no l'has de comptar"
print("✅ Part 1 correcta!")

## Part 2 — Registre: la nota més greu i la més aguda

Implementa `rang_notes`: rep un track i retorna `(nota_minima, nota_maxima)`. Si el track no té cap nota, retorna `(None, None)`.

La faràs servir a la Part 5 per evitar que una transposició surti del rang MIDI.

In [ ]:
def rang_notes(track):
    notes = []
    for msg in track:
        if msg.type == 'note_on' and msg.velocity > 0:
            notes.append(msg.note)
    if len(notes) == 0:          # track sense notes (p. ex. el de metadades): min([]) petaria
        return None, None
    return min(notes), max(notes)

# ── Variants ──
#
# A) Sense llista, amb min/max de dos arguments:
#     minima = 127
#     maxima = 0
#     trobades = 0
#     for msg in track:
#         if msg.type == 'note_on' and msg.velocity > 0:
#             minima = min(minima, msg.note)
#             maxima = max(maxima, msg.note)
#             trobades = trobades + 1
#     if trobades == 0:
#         return None, None
#     return minima, maxima
#
# B) Idiomàtica (comprensió de llista):
#     notes = [msg.note for msg in track if msg.type == 'note_on' and msg.velocity > 0]
#
# ⚠️ PARANYS:
# - Inicialitzar minima = 0 i maxima = 0 (com a C): el mínim sempre surt 0.
# - Inicialitzar a None i fer min(None, nota): TypeError.
# - Amb 127/0 i sense comptar les trobades, el track de metadades retorna (127, 0)
#   en lloc de (None, None): l'autotest ho detecta.
# - min([]) peta amb ValueError: per això cal comprovar la llista buida.

### 🔊 Prova-ho
Compara el registre de la melodia i el del baix. Són els que esperaves en escoltar-lo?

In [ ]:
for i in [1, 2]:
    mn, mx = rang_notes(mid.tracks[i])
    if mn is not None:
        print(f"Track {i} '{mid.tracks[i].name}': de {nom_nota(mn)} ({mn}) a {nom_nota(mx)} ({mx}) — {mx - mn} semitons")

In [ ]:
# AUTOTEST 2
mn, mx = rang_notes(mid.tracks[1])
assert (mn, mx) == (60, 72), f"Melodia: esperava (60, 72), tinc ({mn}, {mx})"
mn, mx = rang_notes(mid.tracks[2])
assert mn is not None and mn < 60, "El baix hauria d'estar per sota del Do central"
assert rang_notes(mid.tracks[0]) == (None, None), "Sense notes → (None, None)"
print("✅ Part 2 correcta!")

## Part 3 — De delta times a temps absolut ⭐

Aquesta és la part important d'avui: passar a codi el que heu fet **en paper** a classe. La farem en dos passos.

### 3a — On és cada missatge?

`msg.time` és el temps **des del missatge anterior**. Per saber en quin tick és cada missatge, cal fer el mateix que a la pissarra: portar un "rellotge" que va sumant.

Completa `ticks_absoluts(track)`. Ha de retornar una llista de parelles `(tick_absolut, msg)`, **una per cada missatge del track** (també els meta: el seu `time` també compta).

Hi ha dues línies per completar. Substitueix cada `pass` per la línia que es demana.

In [ ]:
def ticks_absoluts(track):
    resultat = []
    tick = 0                       # el rellotge comença a zero
    for msg in track:
        tick = tick + msg.time     # 3a.1: el delta s'acumula a TOTS els missatges (també els meta)
        resultat.append((tick, msg))   # 3a.2
    return resultat

### 🔊 Prova-ho: la taula de la pissarra
Aquest és el track que heu fet en paper. Compara la columna **tick abs** amb el que vas escriure tu.

In [ ]:
pissarra = mido.MidiTrack()
pissarra.append(mido.Message('note_on',  note=60, velocity=90,  time=0))
pissarra.append(mido.Message('note_on',  note=64, velocity=90,  time=0))
pissarra.append(mido.Message('note_off', note=60, velocity=0,   time=480))
pissarra.append(mido.Message('note_off', note=64, velocity=0,   time=0))
pissarra.append(mido.Message('note_on',  note=67, velocity=70,  time=240))
pissarra.append(mido.Message('note_off', note=67, velocity=0,   time=240))
pissarra.append(mido.Message('note_on',  note=72, velocity=110, time=0))
pissarra.append(mido.Message('note_off', note=72, velocity=0,   time=960))

print(f"{'missatge':<9} {'nota':>5} {'time':>5} {'tick abs':>9}")
for tick, msg in ticks_absoluts(pissarra):
    print(f"{msg.type:<9} {nom_nota(msg.note):>5} {msg.time:>5} {tick:>9}")

In [ ]:
# AUTOTEST 3a
res = ticks_absoluts(pissarra)
assert len(res) == len(pissarra), f"Hi ha d'haver una parella per missatge: esperava {len(pissarra)}, en tinc {len(res)}"
ticks = []
for tick, msg in res:
    ticks.append(tick)
assert ticks == [0, 0, 480, 480, 720, 960, 960, 1920], \
    f"Ticks incorrectes: {ticks}. Recorda que cada time és des del missatge ANTERIOR"
assert res[4][1] is pissarra[4], "Cada parella ha de portar el seu missatge: (tick, msg)"

res = ticks_absoluts(mid.tracks[1])
assert len(res) == len(mid.tracks[1]), "També cal incloure els missatges meta (track_name, end_of_track...)"
print(f"✅ Part 3a correcta! La melodia acaba al tick {res[-1][0]}")

### 3b — Aparellar cada nota amb el seu final

Una nota són **dos** missatges: un l'obre i un altre la tanca. Ara volem una llista de notes completes `(nota, velocity, inici_s, fi_s)`.

La idea: mentre recorrem els missatges, portem una llista `obertes` amb les notes que **estan sonant**. Quan arriba un missatge que obre una nota, l'hi afegim. Quan arriba un missatge que la tanca, la busquem a `obertes`: així sabem quan havia començat.

La part de buscar ja està feta. Tu has de completar els 5 `TODO`:
- **3b.1** És la mateixa condició que vas escriure a la Part 1.
- **3b.2** Recorda que una nota es pot tancar de **dues** maneres.
- **3b.3** Cada element d'`obertes` és una llista `[nota, velocity, tick_inici]`.
- **3b.4 i 3b.5** Per passar de ticks a segons: `mido.tick2second(ticks, ticks_per_beat, tempo)`.

Pista per a 3b.1 i 3b.2: una variable pot guardar el resultat d'una comparació, per exemple `es_greu = msg.note < 60` val `True` o `False`.

In [ ]:
def a_temps_absolut(track, ticks_per_beat, tempo=500000):
    notes = []
    obertes = []                  # notes que sonen ara. Cada element: [nota, velocity, tick_inici]

    for tick, msg in ticks_absoluts(track):       # la teva funció de la 3a

        obre = msg.type == 'note_on' and msg.velocity > 0                                    # 3b.1
        tanca = msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0)     # 3b.2

        if obre:
            obertes.append([msg.note, msg.velocity, tick])                                   # 3b.3

        elif tanca:
            # Busquem a `obertes` la nota que es tanca (aquesta part ja és feta)
            for i in range(len(obertes)):
                if obertes[i][0] == msg.note:
                    velocity = obertes[i][1]
                    tick_inici = obertes[i][2]
                    inici_s = mido.tick2second(tick_inici, ticks_per_beat, tempo)            # 3b.4
                    fi_s = mido.tick2second(tick, ticks_per_beat, tempo)                     # 3b.5
                    notes.append((msg.note, velocity, inici_s, fi_s))
                    del obertes[i]        # ja no està oberta
                    break                 # només en tanquem una

    return notes

# Errors típics:
# - 3b.2 només amb note_off: el cas de la pissarra (note_on velocity 0) dona 3 notes en lloc de 4.
# - 3b.4/3b.5 amb msg.time en lloc del tick: temps erronis.
# La llista surt en ordre de TANCAMENT, no d'inici: no cal ordenar-la.
# Variant amb diccionari (si algú els coneix): obertes[msg.note] = (vel, tick) i en tancar
# obertes.pop(msg.note). Ull: si la mateixa nota es torna a obrir abans de tancar-se, el diccionari
# sobreescriu la primera i la perd (ho mostra el challenge de l'eco).

### 🔊 Prova-ho
El piano roll es dibuixa **amb la llista que retorna la teva funció**: si la funció està malament, el dibuix també ho estarà.

Després prova de cridar-la amb `tempo=250000` en lloc del tempo del fitxer. Què canvia al dibuix? I per què `escolta()` continua sonant igual?

In [ ]:
tempo_fitxer = tempo_de(mid)
melodia = a_temps_absolut(mid.tracks[1], mid.ticks_per_beat, tempo_fitxer)
baix    = a_temps_absolut(mid.tracks[2], mid.ticks_per_beat, tempo_fitxer)
for nota, vel, inici, fi in melodia[:4]:
    print(f"{nom_nota(nota):>5}  vel {vel:>3}  {inici:6.3f}s → {fi:6.3f}s")
dibuixa_piano_roll([melodia, baix], ['Melodia', 'Baix'], "example_scale.mid — amb la teva funció")
escolta('example_scale.mid')

In [ ]:
# AUTOTEST 3b
def _conte(llista, n):
    # hi ha a la llista una nota igual que n (mateixa nota i velocity, mateixos temps)?
    for x in llista:
        if x[0] == n[0] and x[1] == n[1] and abs(x[2] - n[2]) < 1e-6 and abs(x[3] - n[3]) < 1e-6:
            return True
    return False

def _mateixes(a, b):
    if len(a) != len(b):
        return False
    for n in a:
        if not _conte(b, n):
            return False
    return True

# Referència: pretty_midi fa la mateixa conversió. La fem servir de "segona opinió".
referencia = []
for n in pretty_midi.PrettyMIDI('example_scale.mid').instruments[0].notes:
    referencia.append((n.pitch, n.velocity, n.start, n.end))

res = a_temps_absolut(mid.tracks[1], mid.ticks_per_beat, tempo_fitxer)
assert len(res) == 16, f"Esperava 16 notes, en tinc {len(res)}. Mira 3b.1, 3b.2 i 3b.3"
assert res[1][2] > 0, "Totes les notes comencen a 0 segons: mira 3b.4 i 3b.5"
assert _mateixes(res, referencia), "Els temps de la melodia no coincideixen amb la referència"

# Cas de la pissarra: acord + silenci + tancament amb note_on de velocity 0
t = mido.MidiTrack()
t.append(mido.Message('note_on',  note=60, velocity=90, time=0))
t.append(mido.Message('note_on',  note=64, velocity=90, time=0))
t.append(mido.Message('note_on',  note=67, velocity=90, time=0))
t.append(mido.Message('note_off', note=60, velocity=0,  time=480))
t.append(mido.Message('note_on',  note=64, velocity=0,  time=0))    # tancament amb note_on de velocity 0
t.append(mido.Message('note_off', note=67, velocity=0,  time=0))
t.append(mido.Message('note_on',  note=62, velocity=50, time=480))  # silenci d'una negra abans
t.append(mido.Message('note_off', note=62, velocity=0,  time=240))
esperat = [(60, 90, 0.0, 0.5), (64, 90, 0.0, 0.5), (67, 90, 0.0, 0.5), (62, 50, 1.0, 1.25)]
obtingut = a_temps_absolut(t, 480, 500000)
assert len(obtingut) == 4, \
    f"Amb acord + silenci esperava 4 notes, en tinc {len(obtingut)}. Has tingut en compte el note_on amb velocity 0 a 3b.2?"
assert _mateixes(obtingut, esperat), "Falla amb acord + silenci: revisa els temps (3b.4 i 3b.5)"
print("✅ Part 3 correcta! Ja saps llegir el temps com ho fa un seqüenciador.")

## Part 4 — La teva seqüència (amb un silenci i un acord)

Crea `la_meva_sequencia.mid` amb una frase musical teva. Requisits:
- almenys **8 notes**
- almenys **un silenci** entre notes
- almenys **un acord** (2 o més notes que comencen alhora)
- **velocities variades** (fes que la dinàmica tingui sentit musical)

**Abans d'escriure codi**, escriu la frase en paper com a classe: una taula amb missatge, nota i `time`. El patró dels patches (`note_on` amb `time=0` + `note_off` amb la durada) no et resol ni el silenci ni l'acord: on va el temps del silenci? Quants missatges seguits han de tenir `time=0` en un acord?

In [ ]:
mid_meu = mido.MidiFile(ticks_per_beat=480)
meta = mido.MidiTrack()
meta.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(100), time=0))
mid_meu.tracks.append(meta)

track = mido.MidiTrack()
negra = 480
corxera = 240
# Frase: anacrusi, acord, silenci, línia que creix i torna (com la taula en paper)
track.append(mido.Message('note_on',  note=67, velocity=60, time=0))        # anacrusi Sol
track.append(mido.Message('note_off', note=67, velocity=0,  time=corxera))

track.append(mido.Message('note_on',  note=60, velocity=95, time=0))        # ACORD Do-Mi-Sol:
track.append(mido.Message('note_on',  note=64, velocity=95, time=0))        #   tres note_on seguits
track.append(mido.Message('note_on',  note=67, velocity=95, time=0))        #   amb time=0
track.append(mido.Message('note_off', note=60, velocity=0,  time=negra))    # la durada, al primer tancament
track.append(mido.Message('note_off', note=64, velocity=0,  time=0))
track.append(mido.Message('note_off', note=67, velocity=0,  time=0))

track.append(mido.Message('note_on',  note=65, velocity=55, time=negra))    # SILENCI: el temps va al note_on
track.append(mido.Message('note_off', note=65, velocity=0,  time=corxera))

melodia = [64, 62, 64, 67, 72]
durades = [corxera, corxera, negra, negra, negra * 2]
velocitats = [65, 75, 90, 100, 80]
for i in range(len(melodia)):
    track.append(mido.Message('note_on',  note=melodia[i], velocity=velocitats[i], time=0))
    track.append(mido.Message('note_off', note=melodia[i], velocity=0, time=durades[i]))

track.append(mido.MetaMessage('end_of_track', time=0))
mid_meu.tracks.append(track)
mid_meu.save('la_meva_sequencia.mid')
print("Fitxer creat: la_meva_sequencia.mid")

### 🔊 Prova-ho
Escolta-la i mira-la. Es veuen el silenci i l'acord on els volies? Canvia una velocity i torna-ho a escoltar.

In [ ]:
mm = mido.MidiFile('la_meva_sequencia.mid')
meves = a_temps_absolut(mm.tracks[1], mm.ticks_per_beat, tempo_de(mm))
dibuixa_piano_roll([meves], ['La meva frase'], "La meva seqüència")
escolta('la_meva_sequencia.mid')

In [ ]:
# AUTOTEST 4
mm = mido.MidiFile('la_meva_sequencia.mid')
assert len(mm.tracks) >= 2, "Ha d'haver almenys 2 tracks (metadades + notes)"
n = compta_notes(mm.tracks[1])
assert n is not None, "Aquest autotest fa servir compta_notes: completa primer la Part 1"
assert n >= 8, f"Calen almenys 8 notes, n'hi ha {n}"

# Les notes tal com les sent un sintetitzador (pretty_midi descarta les de durada 0)
sonen = []
for inst in pretty_midi.PrettyMIDI('la_meva_sequencia.mid').instruments:
    for x in inst.notes:
        sonen.append((x.pitch, x.velocity, x.start, x.end))
assert len(sonen) == n, \
    f"Has obert {n} notes però només {len(sonen)} sonen: alguna no es tanca o dura 0 ticks (on has posat el time?)"

velocities_diferents = False
for x in sonen:
    if x[1] != sonen[0][1]:
        velocities_diferents = True
assert velocities_diferents, "Les velocities han de variar"

hi_ha_acord = False
for i in range(len(sonen)):
    for j in range(i + 1, len(sonen)):
        if abs(sonen[i][2] - sonen[j][2]) < 1e-6:
            hi_ha_acord = True
assert hi_ha_acord, "No trobo cap acord: cal que almenys 2 notes comencin al mateix instant"

# Silenci: el final d'alguna nota on no sona res i, més tard, en comença una altra
hi_ha_silenci = False
for a in sonen:
    instant = a[3] + 1e-6
    sona_algo = False
    despres_n_hi_ha = False
    for b in sonen:
        if b[2] <= instant and b[3] > instant:
            sona_algo = True
        if b[2] > instant:
            despres_n_hi_ha = True
    if not sona_algo and despres_n_hi_ha:
        hi_ha_silenci = True
assert hi_ha_silenci, "No trobo cap silenci: cal un moment entre notes on no soni res"
print(f"✅ Part 4 correcta! {n} notes, amb acord i silenci.")

## Part 5 — Transposar

Implementa `transposa(track, semitons)`. Ha de retornar un **track nou** (l'original no s'ha de modificar) on totes les notes estiguin desplaçades `semitons`, tant les que obren com les que tanquen. La resta de missatges es copien igual.
- Si alguna nota sortiria del rang 0–127, la funció ha de retornar `None` (aquí et serveix `rang_notes`).
- Pista: tots els missatges de `mido` tenen `.copy(...)`, que accepta els camps que vols canviar. Per exemple, `msg.copy(note=72)` és el mateix missatge amb la nota 72.

In [ ]:
def transposa(track, semitons):
    minima, maxima = rang_notes(track)
    if minima is not None:
        if minima + semitons < 0 or maxima + semitons > 127:
            return None
    nou = mido.MidiTrack()
    for msg in track:
        if msg.type == 'note_on' or msg.type == 'note_off':
            nou.append(msg.copy(note=msg.note + semitons))
        else:
            nou.append(msg.copy())
    return nou

### 🔊 Prova-ho
Transposa la melodia d'exemple una octava avall i compara-ho amb l'original.

In [ ]:
greu = transposa(mid.tracks[1], -12)
prova = mido.MidiFile(ticks_per_beat=mid.ticks_per_beat)
prova.tracks.append(mid.tracks[0])
prova.tracks.append(greu)
prova.save('melodia_greu.mid')
escolta('example_scale.mid')
escolta('melodia_greu.mid')

In [ ]:
# AUTOTEST 5
original = mid.tracks[1]
notes_abans = []
for m in original:
    if m.type == 'note_on' or m.type == 'note_off':
        notes_abans.append(m.note)

t5 = transposa(original, 7)
assert t5 is not None, "La funció retorna None"

notes_despres = []
for m in original:
    if m.type == 'note_on' or m.type == 'note_off':
        notes_despres.append(m.note)
assert notes_despres == notes_abans, "Has modificat el track original! Cal retornar-ne un de nou"

assert len(t5) == len(original), "El track nou ha de tenir els mateixos missatges"
assert rang_notes(t5) == (67, 79), f"Transposant +7 esperava (67, 79), tinc {rang_notes(t5)}"
for i in range(len(t5)):
    assert t5[i].time == original[i].time, "Els temps no han de canviar"
    if t5[i].type == 'note_off':
        assert t5[i].note == original[i].note + 7, "També has de transposar els missatges que tanquen la nota"
assert transposa(original, 60) is None, "Transposar +60 surt del rang (72 + 60 > 127): ha de retornar None"
print("✅ Part 5 correcta!")

## 🎼 Munta-ho i escolta-ho

Fes un **duo**: un fitxer `duo.mid` amb el track de metadades, la teva frase i una segona veu que sigui la teva frase transposada. Escolta'l i dibuixa les dues veus.

Prova diversos intervals (3, 4, 7, 12, -5...). Quin sona millor amb la teva melodia? Hi ha algun interval per al qual `transposa` retorni `None`? Per què?

In [ ]:
def munta_duo(interval, sortida='duo.mid'):
    mm = mido.MidiFile('la_meva_sequencia.mid')
    segona = transposa(mm.tracks[1], interval)
    if segona is None:
        print(f"No es pot transposar {interval} semitons: surt del rang MIDI")
        return None
    duo = mido.MidiFile(ticks_per_beat=mm.ticks_per_beat)
    duo.tracks.append(mm.tracks[0])
    duo.tracks.append(mm.tracks[1])
    duo.tracks.append(segona)
    duo.save(sortida)
    return duo

for interval in [4, 7, -5]:
    if munta_duo(interval) is not None:
        print(f"Duo a {interval} semitons")
        escolta('duo.mid')

mm = mido.MidiFile('duo.mid')
veus = []
for t in mm.tracks[1:]:
    veus.append(a_temps_absolut(t, mm.ticks_per_beat, tempo_de(mm)))
dibuixa_piano_roll(veus, ['Frase', 'Transposada -5'], "Duo")

munta_duo(60)     # surt del rang: transposa retorna None

---
## 🚀 Challenges (opcionals)

Tots segueixen la mateixa idea, que és com treballa per dins un DAW:

> **deltes → temps absolut → transforma → torna a deltes**

La Part 3 ja et dona la primera fletxa. Per a la tornada, comença per aquí:

**0. `a_track(notes, ticks_per_beat, tempo)`** — el camí invers de la Part 3: rep una llista `(nota, velocity, inici_s, fi_s)` i retorna un `MidiTrack`. Pensa-hi: quan una nota acaba exactament quan en comença una altra, quin missatge ha d'anar primer? Prova-ho amb dues notes iguals seguides.

**1. Eco MIDI — el delay de la S05, ara simbòlic.** `eco(notes, retard_s, repeticions, feedback)`: cada repetició arriba `retard_s` més tard i amb la velocity multiplicada per `feedback`.
- Què passa amb `feedback > 1`? A la S05 el so explotava. I aquí?
- Fes un retard més curt que la durada de les notes. Compara el que en treu la teva `a_temps_absolut` de la Part 3 amb el que en treu `pretty_midi`: quantes notes hi ha i quant duren? Per què no coincideixen? (Pista: quin `note_off` tanca quin `note_on`?)

**2. Retrògrad (la trampa).** Primer prova la solució ingènua: girar la llista de missatges del track, escolta-la i explica què ha passat. Després fes-lo bé: la frase tocada del final al principi, amb el mateix ritme invertit.

**3. Phasing à la Steve Reich (*Piano Phase*).** Dues veus repeteixen el mateix patró de 12 notes, però una fa les notes de 480 ticks i l'altra de 470. Escolta com es desfasen poc a poc. Calcula quantes repeticions calen perquè tornin a coincidir i comprova-ho al piano roll.

**4. Humanitzador.** `humanitza(notes, ms, vel)`: mou cada inici una quantitat aleatòria de ±`ms` i cada velocity ±`vel` (amb `random.randint` o `random.uniform`). Compara la versió rígida amb la humanitzada. A partir de quants mil·lisegons deixa de sonar "humà" i comença a sonar "beguda"?

**5. Accelerando que trenca la Part 3.** Afegeix diversos `set_tempo` progressius al track de metadades. Compara la teva `a_temps_absolut` amb el que fa `pretty_midi`: per què ja no coincideixen? Arregla-la perquè suporti canvis de tempo (pista: `mido.merge_tracks`).

---
# 🚀 Challenges resolts
Totes passen per la Part 3 (`a_temps_absolut`) i pel camí invers (`a_track`). Els `assert` comproven el que descobreix cada challenge.

In [ ]:
import random
TPB = 480
TEMPO = mido.bpm2tempo(120)

def desa(fitxer, tracks, tempo=TEMPO):
    m = mido.MidiFile(ticks_per_beat=TPB)
    meta = mido.MidiTrack()
    meta.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))
    m.tracks.append(meta)
    for t in tracks:
        m.tracks.append(t)
    m.save(fitxer)
    return m

def notes_pretty(fitxer):
    notes = []
    for inst in pretty_midi.PrettyMIDI(fitxer).instruments:
        for n in inst.notes:
            notes.append((n.pitch, n.velocity, n.start, n.end))
    return notes

def mateixes(a, b, tol=1e-3):
    # mateixes notes (en qualsevol ordre), amb tolerància als temps
    if len(a) != len(b):
        return False
    for x in a:
        trobada = False
        for y in b:
            if x[0] == y[0] and x[1] == y[1] and abs(x[2] - y[2]) < tol and abs(x[3] - y[3]) < tol:
                trobada = True
        if not trobada:
            return False
    return True

mid = mido.MidiFile('example_scale.mid')
mel = a_temps_absolut(mid.tracks[1], mid.ticks_per_beat, TEMPO)

## 0. `a_track`: de notes a deltes

In [ ]:
def a_track(notes, ticks_per_beat=TPB, tempo=TEMPO, canal=0):
    # 1) Passem cada nota a dos esdeveniments amb tick absolut
    obrir = []      # [tick, nota, velocity]
    tancar = []     # [tick, nota]
    for nota, vel, inici, fi in notes:
        obrir.append([round(mido.second2tick(inici, ticks_per_beat, tempo)), nota, vel])
        tancar.append([round(mido.second2tick(fi, ticks_per_beat, tempo)), nota])

    # 2) Anem traient esdeveniments per ordre de tick i calculem els deltes
    track = mido.MidiTrack()
    ara = 0
    while len(obrir) > 0 or len(tancar) > 0:
        proper = None                                  # el tick més petit que queda
        for e in obrir + tancar:
            if proper is None or e[0] < proper:
                proper = e[0]
        # Al mateix tick, PRIMER els tancaments i DESPRÉS les obertures:
        # si no, una nota repetida es tancaria just després de tornar-se a obrir
        queden = []
        for e in tancar:
            if e[0] == proper:
                track.append(mido.Message('note_off', channel=canal, note=e[1], velocity=0, time=proper - ara))
                ara = proper
            else:
                queden.append(e)
        tancar = queden
        queden = []
        for e in obrir:
            if e[0] == proper:
                track.append(mido.Message('note_on', channel=canal, note=e[1], velocity=e[2], time=proper - ara))
                ara = proper
            else:
                queden.append(e)
        obrir = queden
    track.append(mido.MetaMessage('end_of_track', time=0))
    return track

repetida = [(60, 80, 0.0, 0.5), (60, 80, 0.5, 1.0), (64, 70, 1.0, 1.5)]
desa('c0.mid', [a_track(repetida)])
assert mateixes(notes_pretty('c0.mid'), repetida), "nota repetida mal ordenada"
assert mateixes(a_temps_absolut(a_track(mel), TPB, TEMPO), mel)
print("✅ anada i tornada correcta, també amb notes repetides")

## 1. Eco MIDI

In [ ]:
def eco(notes, retard_s, repeticions, feedback):
    sortida = []
    for nota, vel, inici, fi in notes:
        for r in range(repeticions + 1):
            v = round(vel * feedback ** r)
            v = min(127, v)                       # feedback > 1: satura, no explota
            if v >= 1:
                sortida.append((nota, v, inici + r * retard_s, fi + r * retard_s))
    return sortida

e = eco(mel, 0.375, 3, 0.6)          # retard (0.375 s) < durada de la nota (0.5 s)
desa('c1_eco.mid', [a_track(e)])
escolta('c1_eco.mid')
vistes = len(a_temps_absolut(mido.MidiFile('c1_eco.mid').tracks[1], TPB, TEMPO))
print(f"Escrites: {len(e)} · pretty_midi en veu: {len(notes_pretty('c1_eco.mid'))} · la Part 3 en veu: {vistes}")

# Mateix nombre de notes... però no les mateixes durades. Mirem el Do4:
part3 = a_temps_absolut(mido.MidiFile('c1_eco.mid').tracks[1], TPB, TEMPO)
for nom, llista in [('Part 3     ', part3), ('pretty_midi', notes_pretty('c1_eco.mid'))]:
    print(nom, end=': ')
    for x in llista:
        if x[0] == 60 and x[2] < 2:
            print(f"{x[2]:.3f}→{x[3]:.3f}", end='  ')
    print()
assert not mateixes(part3, notes_pretty('c1_eco.mid'))

saturat = eco(mel, 0.375, 5, 1.5)
mes_fort = 0
for x in saturat:
    mes_fort = max(mes_fort, x[1])
assert mes_fort == 127

**Descobriments:**
- Amb `feedback > 1` la velocity satura a 127: no explota com el delay de la S05.
- Amb un retard més curt que la nota, la repetició torna a obrir una nota (mateix to i canal) que encara sona. Les dues lectures veuen 64 notes, però **no amb les mateixes durades**: la Part 3 aparella el primer `note_off` amb el primer `note_on` i manté els 0,5 s; `pretty_midi` tanca totes les còpies obertes al primer `note_off`, i els ecos queden tallats (0,375→0,5 s). Si algú fa la Part 3 amb un diccionari, en perd 15 (en veu 49).
- **MIDI no diu quin `note_off` tanca quin `note_on`**: cada receptor ho decideix, i els sintetitzadors reals tampoc no es posen d'acord.

## 2. Retrògrad

In [ ]:
def retrograd_ingenu(track):
    nou = mido.MidiTrack()
    for i in range(len(track) - 1, -1, -1):     # els missatges, de l'últim al primer
        nou.append(track[i])
    return nou

def retrograd(notes):
    total = 0
    for x in notes:
        total = max(total, x[3])
    sortida = []
    for nota, vel, inici, fi in notes:
        sortida.append((nota, vel, total - fi, total - inici))
    return sortida

desa('c2_ingenu.mid', [retrograd_ingenu(mid.tracks[1])])
print(f"Ingenu: només {len(notes_pretty('c2_ingenu.mid'))} de 16 notes sonen")
escolta('c2_ingenu.mid')

r = retrograd(mel)
assert mateixes(retrograd(r), mel)            # el retrògrad del retrògrad és l'original
desa('c2_retrograd.mid', [a_track(r)])
escolta('c2_retrograd.mid')

**Per què falla l'ingenu:** cada `note_off` queda abans del seu `note_on`, i cada delta queda enganxat al missatge equivocat. Cal passar pel temps absolut.

## 3. Phasing (Reich, *Piano Phase*)

In [ ]:
PATRO_REICH = [64, 66, 71, 73, 74, 66, 64, 73, 71, 66, 74, 73]

def veu_repetida(patro, ticks, repeticions, canal, vel=80):
    t = mido.MidiTrack()
    for r in range(repeticions):
        for n in patro:
            t.append(mido.Message('note_on',  channel=canal, note=n, velocity=vel, time=0))
            t.append(mido.Message('note_off', channel=canal, note=n, velocity=0,  time=ticks))
    t.append(mido.MetaMessage('end_of_track', time=0))
    return t

a = veu_repetida(PATRO_REICH, 480, 48, 0)
b = veu_repetida(PATRO_REICH, 470, 48, 1)
pa = a_temps_absolut(a, TPB, TEMPO)
pb = a_temps_absolut(b, TPB, TEMPO)
assert abs(pa[47][2] - pb[48][2]) < 1e-9      # després de 48 notes, B va exactament una nota per davant

a8 = veu_repetida(PATRO_REICH, 480, 8, 0)     # per escoltar n'hi ha prou amb 8 repeticions
b8 = veu_repetida(PATRO_REICH, 470, 8, 1)
desa('c3_phasing.mid', [a8, b8], tempo=mido.bpm2tempo(480))
escolta('c3_phasing.mid')
dibuixa_piano_roll([a_temps_absolut(a8, TPB, mido.bpm2tempo(480)),
                    a_temps_absolut(b8, TPB, mido.bpm2tempo(480))],
                   ['480 ticks', '470 ticks'], "Phasing: 8 repeticions")

**El càlcul:** cada nota, B s'avança 480 − 470 = 10 ticks. Torna a caure a temps (una nota per davant) cada 480 / 10 = **48 notes = 4 repeticions**. El patró sencer torna a coincidir quan B s'ha avançat 12 notes: 12 × 48 = 576 notes = **48 repeticions**.

## 4. Humanitzador

In [ ]:
def humanitza(notes, ms=15, vel=10):
    sortida = []
    for nota, v, inici, fi in notes:
        nou_inici = inici + random.uniform(-ms, ms) / 1000
        if nou_inici < 0:
            nou_inici = 0.0
        nova_v = v + random.randint(-vel, vel)
        nova_v = max(1, min(127, nova_v))
        sortida.append((nota, nova_v, nou_inici, nou_inici + (fi - inici)))
    return sortida

random.seed(1)                       # perquè surti igual cada vegada
for ms in [0, 20, 80]:
    desa(f'c4_huma_{ms}.mid', [a_track(humanitza(mel, ms=ms, vel=12))])
    print(f"±{ms} ms")
    escolta(f'c4_huma_{ms}.mid')

## 5. Accelerando

In [ ]:
def accelerando(bpm_inici=60, bpm_final=180, beats=16):
    meta = mido.MidiTrack()
    for b in range(beats):
        bpm = bpm_inici + (bpm_final - bpm_inici) * b / (beats - 1)
        delta = TPB
        if b == 0:
            delta = 0
        meta.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(bpm), time=delta))
    meta.append(mido.MetaMessage('end_of_track', time=0))
    t = mido.MidiTrack()
    for b in range(beats):
        t.append(mido.Message('note_on',  note=60 + b % 8, velocity=80, time=0))
        t.append(mido.Message('note_off', note=60 + b % 8, velocity=0,  time=TPB))
    t.append(mido.MetaMessage('end_of_track', time=0))
    return meta, t

def a_temps_absolut_tempomap(mid):
    # Com la Part 3, però cada delta es converteix amb el tempo VIGENT en aquell moment.
    # merge_tracks ajunta tots els tracks (amb els deltes recalculats) per veure els set_tempo.
    tempo = 500000
    segons = 0.0
    obertes = []                      # [canal, nota, velocity, inici_s]
    notes = []
    for msg in mido.merge_tracks(mid.tracks):
        segons = segons + mido.tick2second(msg.time, mid.ticks_per_beat, tempo)
        if msg.type == 'set_tempo':
            tempo = msg.tempo
        elif msg.type == 'note_on' and msg.velocity > 0:
            obertes.append([msg.channel, msg.note, msg.velocity, segons])
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            for i in range(len(obertes)):
                if obertes[i][0] == msg.channel and obertes[i][1] == msg.note:
                    notes.append((msg.note, obertes[i][2], obertes[i][3], segons))
                    del obertes[i]
                    break
    return notes

meta, t = accelerando()
acc = mido.MidiFile(ticks_per_beat=TPB)
acc.tracks.append(meta)
acc.tracks.append(t)
acc.save('c5_accel.mid')
escolta('c5_accel.mid')

ingenua = a_temps_absolut(t, TPB, TEMPO)
assert not mateixes(ingenua, notes_pretty('c5_accel.mid'))        # la Part 3 no hi arriba
assert mateixes(a_temps_absolut_tempomap(acc), notes_pretty('c5_accel.mid'), tol=1e-6)
print(f"Durada real {acc.length:.2f} s vs {ingenua[-1][3]:.2f} s amb tempo fix")
dibuixa_piano_roll([ingenua, a_temps_absolut_tempomap(acc)],
                   ['Tempo fix (Part 3)', 'Amb mapa de tempo'], "Accelerando 60→180 bpm")
print("✅ challenges: tot correcte")